# Deploy MiniMax-M2.5 on Azure ML Managed Online Endpoint

**Model:** `MiniMaxAI/MiniMax-M2.5` (230B total, 10B activated MoE)  
**VM:** `Standard_NC96ads_A100_v4` — 4× NVIDIA A100 80GB (PCIe)  
**Serving:** vLLM with TP4 (tensor-parallel-size 4)  
**Image:** `vllm/vllm-openai:latest`

**A100 PCIe workarounds applied:**
- `NCCL_P2P_DISABLE=1` — no NVLink, route all-reduce via host memory
- `--enforce-eager` — disable CUDA graphs (require P2P)
- `VLLM_DISABLED_KERNELS=TritonFp8BlockScaledMMKernel` — Triton FP8 E4M3 needs Hopper (sm_89+)

This notebook orchestrates the full deployment lifecycle:
1. Install & import dependencies
2. Connect to AML workspace
3. Configuration
4. Build Docker image in ACR
5. Create managed online endpoint
6. Attach ACR to workspace (one-time setup)
7. Deploy MiniMax-M2.5 with TP4
8. Set traffic to 100%
9. Get endpoint credentials
10. Test — chat completion & streaming
11. Performance benchmark
12. Check deployment logs
13. Cleanup (optional)

## 1. Install & Import Dependencies

In [ ]:
%pip install azure-ai-ml azure-identity openai python-dotenv --quiet

In [ ]:
import json
import os
import requests
from dotenv import load_dotenv
from azure.ai.ml import MLClient, load_environment, load_online_endpoint, load_online_deployment
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

load_dotenv()  # Load .env file

## 2. Connect to AML Workspace

In [ ]:
required_vars = ["AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_WORKSPACE_NAME", "AZURE_TENANT_ID"]
missing = [v for v in required_vars if not os.environ.get(v)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}\nCreate a .env file from .env.example")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP"]
workspace_name = os.environ["AZURE_WORKSPACE_NAME"]
tenant_id = os.environ["AZURE_TENANT_ID"]

try:
    credential = DefaultAzureCredential(additionally_allowed_tenants=[tenant_id])
    credential.get_token("https://management.azure.com/.default", tenant_id=tenant_id)
except Exception:
    credential = InteractiveBrowserCredential(tenant_id=tenant_id)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)
print(f"Connected to workspace: {ml_client.workspace_name}")

## 3. Configuration

Define all deployment parameters in one place.

In [ ]:
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "minimax-m25-prod")
DEPLOYMENT_NAME = os.environ.get("DEPLOYMENT_NAME", "current")
MODEL_NAME = os.environ.get("MODEL_NAME", "MiniMaxAI/MiniMax-M2.5")

print(f"Endpoint:  {ENDPOINT_NAME}")
print(f"Model:     {MODEL_NAME}")

## 4. Build Docker Image in ACR

The workspace storage account has key-based auth disabled, so we build the Docker image directly in ACR using `az acr build` (bypasses workspace blob storage entirely).

In [ ]:
import subprocess

CONFIG_DIR = "config_nca100-minimax25"

# Step 1: Get the workspace ACR name
ws = ml_client.workspaces.get(workspace_name)
acr_id = ws.container_registry

if not acr_id:
    result = subprocess.run(
        f'az acr list --resource-group {resource_group} --query "[].name" -o json',
        capture_output=True, text=True, shell=True,
    )
    acrs = json.loads(result.stdout) if result.returncode == 0 else []
    if acrs:
        acr_name = acrs[0]
        print(f"Workspace has no ACR. Using ACR from resource group: {acr_name}")
    else:
        acr_name = workspace_name.replace("_", "") + "acr"
        print(f"No ACR found. Creating '{acr_name}' in {resource_group}...")
        result = subprocess.run(
            f'az acr create --name {acr_name} --resource-group {resource_group} --sku Basic --admin-enabled true',
            capture_output=True, text=True, shell=True,
        )
        if result.returncode != 0:
            print("STDERR:", result.stderr[-2000:])
            raise RuntimeError(f"ACR creation failed")
        print(f"ACR '{acr_name}' created.")
else:
    acr_name = acr_id.split("/")[-1]

print(f"Using ACR: {acr_name}")

# Step 2: Build the Docker image directly in ACR
image_tag = f"{acr_name}.azurecr.io/minimax-m25-vllm:latest"
cmd = f'az acr build --registry {acr_name} --image minimax-m25-vllm:latest ./{CONFIG_DIR}'
print(f"Building image: {image_tag}")
print("This may take a few minutes...")

result = subprocess.run(cmd, capture_output=True, text=True, shell=True)

if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"ACR build failed (exit code {result.returncode})")

print(f"Image built successfully: {image_tag}")

## 5. Create Managed Online Endpoint

In [ ]:
endpoint = load_online_endpoint(source=f"{CONFIG_DIR}/endpoint.yml")
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint '{endpoint.name}' created (provisioning state: {endpoint.provisioning_state})")
print(f"Scoring URI: {endpoint.scoring_uri}")

## 6. Attach ACR to Workspace (one-time setup)

Attaches the ACR to the workspace so AML auto-wires `AcrPull` permissions for the endpoint's managed identity. Also deletes and recreates the endpoint so the new identity gets the permissions.

> **Run this cell only once** when setting up a new workspace/ACR combination. Skip on subsequent deployments.

In [ ]:
# Attach the ACR to the workspace so AML auto-wires pull permissions
acr_resource_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.ContainerRegistry/registries/{acr_name}"
cmd = (
    f'az ml workspace update'
    f' --name {workspace_name}'
    f' --resource-group {resource_group}'
    f' --container-registry {acr_resource_id}'
    f' --update-dependent-resources'
)
result = subprocess.run(cmd, capture_output=True, text=True, shell=True)

if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
    raise RuntimeError("Failed to attach ACR to workspace")

print(f"ACR '{acr_name}' attached to workspace '{workspace_name}'")

# Recreate endpoint so its managed identity gets AcrPull auto-wired
print("Deleting and recreating endpoint to refresh identity permissions...")
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Old endpoint deleted.")

## 7. Deploy MiniMax-M2.5 with TP4

Deploys on `Standard_NC96ads_A100_v4` (4× A100 80GB PCIe). Liveness/readiness probes have **1800s (30 min) initial delay** for model download (~230 GB FP8) and GPU loading. Takes **30-60+ minutes**.

In [ ]:
deployment = load_online_deployment(source=f"{CONFIG_DIR}/deployment.yml")
deployment.environment.image = image_tag

print(f"Starting deployment '{deployment.name}' on {deployment.instance_type}...")
print(f"Image: {image_tag}")
print("This will take 30-60+ minutes (image pull + model download + GPU loading)")
deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment provisioning state: {deployment.provisioning_state}")

## 8. Set Traffic to 100%

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic set to 100% on '{DEPLOYMENT_NAME}'")

## 9. Get Endpoint Credentials

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
keys = ml_client.online_endpoints.get_keys(ENDPOINT_NAME)

scoring_uri = endpoint.scoring_uri
api_key = keys.primary_key

print(f"Scoring URI: {scoring_uri}")
print(f"API Key:     {api_key[:8]}...")

## 10. Test — Chat Completion (requests)

In [ ]:
base_url = scoring_uri.rstrip("/")
url = f"{base_url}/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}",
}
payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Explain what a Mixture-of-Experts model is in 3 sentences."},
    ],
    "max_tokens": 256,
}

response = requests.post(url, headers=headers, json=payload)
result = response.json()
print(json.dumps(result, indent=2))

## 10b. Test — Streaming with OpenAI SDK

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{base_url}/v1",
    api_key=api_key,
)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a Python function to compute the Fibonacci sequence using memoization."},
    ],
    max_tokens=512,
    stream=True,
)

for chunk in response:
    delta = chunk.choices[0].delta
    if hasattr(delta, "content") and delta.content:
        print(delta.content, end="", flush=True)
print()  # newline at end

## 11. Performance Benchmark

Sends concurrent streaming requests with ~2048 input tokens and measures throughput, TTFT, TPOT, and ITL. Uses server-reported `usage` for accurate token counting. First 5 requests are warmup and excluded from stats.

In [ ]:
import asyncio
import time
import random
import string
import statistics
from openai import AsyncOpenAI

# Benchmark config
NUM_PROMPTS = 25         # total requests (first 5 are warmup, discarded)
WARMUP = 5               # warmup requests to discard from stats
MAX_CONCURRENCY = 5      # concurrent requests
MAX_OUTPUT_TOKENS = 256  # max output per request

# Generate a prompt with a known token count using repeated real words
# Real English words tokenize more predictably than random chars (~1 token per word)
WORD_POOL = "the quick brown fox jumps over a lazy dog while running through fields of green grass under bright blue skies with warm summer winds blowing gently across the open meadow".split()

def make_prompt(target_tokens=2048):
    """Generate a prompt targeting ~target_tokens input tokens using real words."""
    # ~1 token per word for common English, plus overhead for the instruction
    words = [random.choice(WORD_POOL) for _ in range(target_tokens)]
    return f"Summarize the following text in one paragraph:\n\n{' '.join(words)}"

async def bench_one(client, prompt_id, prompt):
    """Send one streaming request, measure TTFT, TPOT, and token count accurately."""
    t0 = time.perf_counter()
    ttft = None
    token_times = []  # timestamp of each content-bearing chunk
    usage_tokens = None

    stream = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_OUTPUT_TOKENS,
        stream=True,
        stream_options={"include_usage": True},  # get accurate token count in final chunk
    )
    async for chunk in stream:
        now = time.perf_counter()
        # Only count chunks with actual content (skip empty role/reasoning chunks)
        if chunk.choices and chunk.choices[0].delta.content:
            if ttft is None:
                ttft = now - t0  # TTFT = time to first *content* token
            token_times.append(now)
        # Capture usage from the final chunk (accurate server-side token count)
        if hasattr(chunk, 'usage') and chunk.usage is not None:
            usage_tokens = chunk.usage.completion_tokens

    total_time = time.perf_counter() - t0

    # Use server-reported token count if available, otherwise fall back to chunk count
    generated_tokens = usage_tokens if usage_tokens else len(token_times)

    # Compute inter-token latencies from timestamps
    itls = []
    for i in range(1, len(token_times)):
        itls.append((token_times[i] - token_times[i-1]) * 1000)  # ms

    # TPOT = (total_time - ttft) / (tokens - 1) if we have enough tokens
    tpot = None
    if ttft and generated_tokens > 1:
        tpot = (total_time - ttft) / (generated_tokens - 1) * 1000  # ms

    return {
        "id": prompt_id,
        "ttft_ms": ttft * 1000 if ttft else None,
        "total_s": total_time,
        "tokens": generated_tokens,
        "tpot_ms": tpot,
        "itls": itls,
    }

async def run_benchmark():
    client = AsyncOpenAI(base_url=f"{base_url}/v1", api_key=api_key)
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    prompts = [make_prompt() for _ in range(NUM_PROMPTS)]

    async def limited(i, p):
        async with semaphore:
            try:
                return await bench_one(client, i, p)
            except Exception as e:
                return {"id": i, "error": str(e)}

    print(f"Benchmarking: {NUM_PROMPTS} prompts ({WARMUP} warmup + {NUM_PROMPTS - WARMUP} measured)")
    print(f"  Concurrency: {MAX_CONCURRENCY}, Max output tokens: {MAX_OUTPUT_TOKENS}")
    print(f"  Input: ~2048 tokens per prompt (real English words)")
    print("Running...")

    t_start = time.perf_counter()
    results = await asyncio.gather(*[limited(i, p) for i, p in enumerate(prompts)])
    t_total = time.perf_counter() - t_start

    # Separate warmup from measured results
    all_ok = [r for r in results if "error" not in r]
    failed = [r for r in results if "error" in r]

    # Sort by request id, discard first WARMUP
    all_ok.sort(key=lambda r: r["id"])
    warmup_results = all_ok[:WARMUP]
    measured = all_ok[WARMUP:]

    ttfts = [r["ttft_ms"] for r in measured if r["ttft_ms"] is not None]
    tpots = [r["tpot_ms"] for r in measured if r["tpot_ms"] is not None]
    all_itls = [itl for r in measured for itl in r["itls"]]
    total_output_tokens = sum(r["tokens"] for r in measured)
    measured_duration = sum(r["total_s"] for r in measured) / MAX_CONCURRENCY  # effective wall time

    print(f"\n{'='*55}")
    print(f"  Benchmark Results (excluding {WARMUP} warmup requests)")
    print(f"{'='*55}")
    print(f"  Successful requests:          {len(measured)}")
    print(f"  Failed requests:              {len(failed)}")
    print(f"  Total benchmark duration (s): {t_total:.1f}")
    print(f"  Total generated tokens:       {total_output_tokens}")
    print(f"  Request throughput (req/s):    {len(measured)/t_total:.2f}")
    print(f"  Output token throughput (t/s): {total_output_tokens/t_total:.1f}")

    def print_stats(name, values):
        if len(values) < 2:
            print(f"\n  --- {name} ---")
            print(f"  Insufficient data ({len(values)} samples)")
            return
        values.sort()
        p50_idx = len(values) // 2
        p99_idx = min(int(len(values) * 0.99), len(values) - 1)
        print(f"\n  --- {name} (n={len(values)}) ---")
        print(f"  Mean:   {statistics.mean(values):.0f} ms")
        print(f"  Median: {values[p50_idx]:.0f} ms")
        print(f"  Stdev:  {statistics.stdev(values):.0f} ms")
        print(f"  P99:    {values[p99_idx]:.0f} ms")
        print(f"  Min:    {values[0]:.0f} ms")
        print(f"  Max:    {values[-1]:.0f} ms")

    print_stats("Time to First Token (TTFT)", ttfts)
    print_stats("Time per Output Token (TPOT)", tpots)
    print_stats("Inter-Token Latency (ITL)", all_itls)

    if failed:
        print(f"\n  Errors: {[r['error'][:100] for r in failed[:3]]}")

    # Sanity check: compare chunk-based vs server-reported tokens
    usage_available = sum(1 for r in measured if r["tokens"] > 0)
    print(f"\n  Token counting: server-reported usage in {usage_available}/{len(measured)} responses")

await run_benchmark()

## 12. Check Deployment Logs (for debugging)

In [ ]:
logs = ml_client.online_deployments.get_logs(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    lines=100,
)
print(logs)

## 13. Cleanup (Optional)

**Only run this when you're done with the POC.** This deletes the endpoint and deployment to stop billing.

In [ ]:
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")